In [ ]:
#@title ARL — is every addition worth it? (run this one cell)
# Runtime > Change runtime type > A100 GPU (High-RAM). Everything (checkpoints, logs, metrics, report) is saved
# under RESULTS on Google Drive and the cell resumes where it stopped if Colab disconnects: just run it again.
MODE     = 'quick'   # 'quick': 2 variants x 2 epochs, 1 seed, checks the whole pipeline | 'full': the study
SEEDS    = 3         # 'full' only: training seeds per variant (each variant also trains 1 validation model)
VARIANTS = 'all'     # 'full' only: 'all' (9 variants) | 'core' (base, A1, A2, A3, A1A2A3) | e.g. 'core avg'
PRECISION = '32-true'  # as TrackTacular; 'bf16-mixed' is faster on an A100
RESULTS  = '/content/drive/MyDrive/retrack_results'
REPO     = 'github.com/rizwanmustafa/retrack.git'
BRANCH   = 'feat/arl-attempt-1'

import os, glob
from google.colab import drive, userdata
drive.mount('/content/drive')

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

# Optional Colab Secrets (key icon on the left): KAGGLE_USERNAME / KAGGLE_KEY if the WILDTRACK download is refused,
# GITHUB_TOKEN if the repository is private.
for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
    if secret(k):
        os.environ[k] = secret(k)
url = f"https://{secret('GITHUB_TOKEN')}@{REPO}" if secret('GITHUB_TOKEN') else f'https://{REPO}'
if not os.path.exists('/content/retrack'):
    !git clone -q -b {BRANCH} {url} /content/retrack
else:
    !git -C /content/retrack fetch -q origin {BRANCH} && git -C /content/retrack checkout -q {BRANCH} && git -C /content/retrack pull -q origin {BRANCH}

extra = f'--precision {PRECISION}'
if MODE == 'full':
    extra += f' --seeds {SEEDS} --variants {VARIANTS}'
!PIPELINE=arl bash /content/retrack/colab/run.sh {MODE} {RESULTS} {extra}

from IPython.display import Markdown, Image, display
out = f'{RESULTS}/arl_{MODE}'
if os.path.exists(f'{out}/report/REPORT.md'):
    display(Markdown(open(f'{out}/report/REPORT.md').read()))
    if os.path.exists(f'{out}/report/additions.png'):
        display(Image(f'{out}/report/additions.png', width=720))
print('Everything for this run:', out)
print('Best checkpoints:        ', f'{out}/best')
print('Share for analysis:      ', f'{out}/for_claude.zip  (everything except checkpoints)')
